# 🤖 Certificate Classification - Part 3: Model Training

## Goal
Train multiple ML models to classify certificates as malicious/benign.

### Models We'll Train
1. **Logistic Regression** - Simple baseline
2. **Random Forest** - Ensemble of decision trees
3. **Gradient Boosting** - Sequential boosting
4. **XGBoost** - Optimized gradient boosting (if installed)
5. **LightGBM** - Fast gradient boosting (if installed)
6. **Neural Network (MLP)** - Deep learning baseline

### Handling Class Imbalance
- Class weights
- SMOTE oversampling
- Stratified splits

---

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, f1_score
)

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Optional: XGBoost, LightGBM
try:
    import xgboost as xgb
    HAS_XGB = True
    print('✅ XGBoost available')
except ImportError:
    HAS_XGB = False
    print('⚠️ XGBoost not installed (pip install xgboost)')

try:
    import lightgbm as lgb
    HAS_LGB = True
    print('✅ LightGBM available')
except ImportError:
    HAS_LGB = False
    print('⚠️ LightGBM not installed (pip install lightgbm)')

# Visualization
import matplotlib.pyplot as plt

## 2. Load Processed Data

In [ ]:
data_dir = Path('./outputs/ml')

# Try loading pickled features first
if (data_dir / 'X_features.pkl').exists():
    X = pd.read_pickle(data_dir / 'X_features.pkl')
    y = pd.read_pickle(data_dir / 'y_target.pkl')
    print('Loaded from pickle files')
else:
    print('❌ Run 02_feature_engineering.ipynb first!')
    raise FileNotFoundError('X_features.pkl not found')

print(f'Features: {X.shape}')
print(f'Target: {y.shape}')
print(f'Positive rate: {y.mean()*100:.4f}%')

## 3. Train/Test Split

In [ ]:
# Stratified split to maintain class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f'Train: {X_train.shape[0]:,} samples ({y_train.sum()} positive)')
print(f'Test:  {X_test.shape[0]:,} samples ({y_test.sum()} positive)')

In [ ]:
# Scale features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Calculate class weight for imbalanced data
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f'Class weight (neg/pos): {scale_pos_weight:.1f}')

## 4. Define Evaluation Function

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, results_dict):
    """
    Train and evaluate a model, store results.
    """
    print(f'\n{"="*60}')
    print(f'🔹 {name}')
    print(f'{"="*60}')
    
    # Train
    model.fit(X_tr, y_tr)
    
    # Predict
    y_pred = model.predict(X_te)
    
    # Probabilities (if available)
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_te)[:, 1]
    else:
        y_proba = y_pred.astype(float)
    
    # Metrics
    roc_auc = roc_auc_score(y_te, y_proba)
    ap = average_precision_score(y_te, y_proba)
    f1 = f1_score(y_te, y_pred)
    
    print(f'\nROC-AUC: {roc_auc:.4f}')
    print(f'Average Precision: {ap:.4f}')
    print(f'F1 Score: {f1:.4f}')
    
    print('\nClassification Report:')
    print(classification_report(y_te, y_pred, digits=4))
    
    print('Confusion Matrix:')
    cm = confusion_matrix(y_te, y_pred)
    print(cm)
    
    # Store results
    results_dict[name] = {
        'model': model,
        'roc_auc': roc_auc,
        'avg_precision': ap,
        'f1': f1,
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    
    return model

## 5. Train Models

In [ ]:
# Store all results
results = {}

### 5.1 Logistic Regression

In [ ]:
lr = LogisticRegression(
    max_iter=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

evaluate_model('Logistic Regression', lr, X_train_scaled, X_test_scaled, y_train, y_test, results)

### 5.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

evaluate_model('Random Forest', rf, X_train, X_test, y_train, y_test, results)

### 5.3 Gradient Boosting

In [ ]:
# Note: sklearn GradientBoosting doesn't support class_weight
# We'll use sample_weight instead

sample_weights = np.where(y_train == 1, scale_pos_weight, 1.0)

gb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

# Custom training with sample weights
print('\n' + '='*60)
print('🔹 Gradient Boosting')
print('='*60)

gb.fit(X_train, y_train, sample_weight=sample_weights)
y_pred = gb.predict(X_test)
y_proba = gb.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred)

print(f'\nROC-AUC: {roc_auc:.4f}')
print(f'Average Precision: {ap:.4f}')
print(f'F1 Score: {f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

results['Gradient Boosting'] = {
    'model': gb,
    'roc_auc': roc_auc,
    'avg_precision': ap,
    'f1': f1,
    'y_pred': y_pred,
    'y_proba': y_proba
}

### 5.4 XGBoost (if available)

In [ ]:
if HAS_XGB:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    
    evaluate_model('XGBoost', xgb_clf, X_train, X_test, y_train, y_test, results)
else:
    print('⏭️ Skipping XGBoost (not installed)')

### 5.5 LightGBM (if available)

In [ ]:
if HAS_LGB:
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    evaluate_model('LightGBM', lgb_clf, X_train, X_test, y_train, y_test, results)
else:
    print('⏭️ Skipping LightGBM (not installed)')

### 5.6 Neural Network (MLP)

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

evaluate_model('Neural Network (MLP)', mlp, X_train_scaled, X_test_scaled, y_train, y_test, results)

## 6. Model Comparison Summary

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'ROC-AUC': [r['roc_auc'] for r in results.values()],
    'Avg Precision': [r['avg_precision'] for r in results.values()],
    'F1 Score': [r['f1'] for r in results.values()]
}).sort_values('ROC-AUC', ascending=False)

print('\n' + '='*60)
print('📊 MODEL COMPARISON')
print('='*60)
print(comparison.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison))
width = 0.25

ax.bar(x - width, comparison['ROC-AUC'], width, label='ROC-AUC', color='steelblue')
ax.bar(x, comparison['Avg Precision'], width, label='Avg Precision', color='darkorange')
ax.bar(x + width, comparison['F1 Score'], width, label='F1 Score', color='green')

ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 7. Feature Importance (Best Model)

In [ ]:
# Get best model (by ROC-AUC)
best_name = comparison.iloc[0]['Model']
best_model = results[best_name]['model']

print(f'Best model: {best_name}')

# Extract feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X.columns.tolist()
    
    feat_imp = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print('\nTop 20 Features:')
    print(feat_imp.head(20).to_string(index=False))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = feat_imp.head(20)
    ax.barh(top_n['feature'], top_n['importance'], color='steelblue')
    ax.set_xlabel('Importance')
    ax.set_title(f'Top 20 Feature Importances ({best_name})')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('(Feature importance not available for this model type)')

## 8. Save Best Model

In [ ]:
import pickle

output_dir = Path('./outputs/ml')

# Save best model
with open(output_dir / 'best_model.pkl', 'wb') as f:
    pickle.dump({
        'name': best_name,
        'model': best_model,
        'scaler': scaler,
        'feature_names': X.columns.tolist(),
        'metrics': {
            'roc_auc': results[best_name]['roc_auc'],
            'avg_precision': results[best_name]['avg_precision'],
            'f1': results[best_name]['f1']
        }
    }, f)

# Save all results
comparison.to_csv(output_dir / 'model_comparison.csv', index=False)

print(f'✅ Saved best model: {best_name}')
print(f'   - best_model.pkl')
print(f'   - model_comparison.csv')